# Resolução do Roteiro Prático - Conexão Neon e Pandas

Este notebook contém a resolução de todos os exercícios de manipulação de dados com Pandas (`.loc`, `.iloc`, `groupby`, `merge` e `broadcasting`) conectados ao banco PostgreSQL no Neon.

## 1. Conexão com o Banco de Dados

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from google.colab import userdata

# Carregando a URL de conexão do Secrets do Colab
DATABASE_URL = userdata.get('DB_URL')
engine = create_engine(DATABASE_URL)

# Lendo a tabela de vendas
df_vendas = pd.read_sql("SELECT * FROM vendas_loja;", engine)
df_vendas.head()

## 2. Parte 1: Exercícios com .loc e .iloc

### Exercício 1 (Filtro Condicional e Atribuição com .loc)
Calcular a coluna de faturamento (`quantidade * valor_unitario`) e filtrar apenas vendas de 'SP' com faturamento > R$ 500,00.

In [ ]:
# Criando a coluna de faturamento
df_vendas['faturamento'] = df_vendas['quantidade'] * df_vendas['valor_unitario']

# Filtro usando .loc
ex1 = df_vendas.loc[(df_vendas['estado_cliente'] == 'SP') & (df_vendas['faturamento'] > 500)]
ex1

### Exercício 2 (Seleção de Colunas Específicas com .loc)
Filtrar categoria 'Eletrônicos', exibindo apenas `cliente`, `produto` e `faturamento`.

In [ ]:
ex2 = df_vendas.loc[df_vendas['categoria'] == 'Eletrônicos', ['cliente', 'produto', 'faturamento']]
ex2.head()

### Exercício 3 (Modificação de Valores com .loc)
Aplicar desconto de 10% no `valor_unitario` de todas as vendas da categoria 'Acessórios'.

In [ ]:
# Aplicando desconto com .loc
df_vendas.loc[df_vendas['categoria'] == 'Acessórios', 'valor_unitario'] *= 0.90

# Recalculando faturamento das linhas afetadas
df_vendas['faturamento'] = df_vendas['quantidade'] * df_vendas['valor_unitario']

# Verificando o resultado
df_vendas.loc[df_vendas['categoria'] == 'Acessórios'].head()

### Exercício 4 (Fatiamento por Posição com .iloc)
Primeiras 5 linhas e colunas dos índices 1 a 4 (`data_venda` até `categoria`).

In [ ]:
ex4 = df_vendas.iloc[0:5, 1:5]
ex4

### Exercício 5 (Acesso à Célula Específica com .iloc)
Extrair o nome do cliente da primeira venda registrada (linha 0).

In [ ]:
# Assumindo que a coluna 'cliente' está na posição/índice 2
primeiro_cliente = df_vendas.iloc[0, df_vendas.columns.get_loc('cliente')]
print(f"Nome do primeiro cliente: {primeiro_cliente}")

### Exercício 6 (Fatiamento Inverso com .iloc)
Últimos 10 registros e últimas 3 colunas.

In [ ]:
ex6 = df_vendas.iloc[-10:, -3:]
ex6

### Desafio Integrado da Parte 1
1. Extração via `read_sql`
2. Transformação de `faturamento`
3. Filtro com `.loc` (Categorias 'Móveis'/'Informática' em 'RJ'/'MG')
4. Ordenação e exibição dos Top 5 com `.iloc`.

In [ ]:
# 1 e 2. Extração e Transformação
df_desafio = pd.read_sql("SELECT * FROM vendas_loja;", engine)
df_desafio['faturamento'] = df_desafio['quantidade'] * df_desafio['valor_unitario']

# 3. Filtro com .loc
filtro_loc = df_desafio.loc[
    (df_desafio['categoria'].isin(['Móveis', 'Informática'])) &
    (df_desafio['estado_cliente'].isin(['RJ', 'MG']))
]

# 4. Ordenação e Top 5 via .iloc
filtro_ordenado = filtro_loc.sort_values(by='faturamento', ascending=False)
resultado_desafio = filtro_ordenado.iloc[0:5][['cliente', 'produto', 'faturamento']]
resultado_desafio

## 3. Parte 2: GroupBy, Merge e Broadcasting

### Exercício 7 (Agrupamento com GroupBy)
- Faturamento total por categoria
- Ticket médio por estado

In [ ]:
# Faturamento total por categoria
fat_categoria = df_vendas.groupby('categoria')['faturamento'].sum().reset_index()
print("--- Faturamento por Categoria ---")
print(fat_categoria)

# Ticket médio (média do faturamento) por estado
ticket_estado = df_vendas.groupby('estado_cliente')['faturamento'].mean().reset_index()
ticket_estado.columns = ['estado_cliente', 'ticket_medio']
print("\n--- Ticket Médio por Estado ---")
print(ticket_estado.head())

### Exercício 8 (Junção de Dados com Merge)
Carregar a tabela `metas_estados` e unir com os faturamentos acumulados por estado.

In [ ]:
# Lendo a nova tabela de metas
df_metas = pd.read_sql("SELECT * FROM metas_estados;", engine)

# Agrupando o faturamento total por estado
vendas_por_estado = df_vendas.groupby('estado_cliente')['faturamento'].sum().reset_index()
vendas_por_estado.columns = ['estado_cliente', 'faturamento_real']

# Realizando o Merge
df_merge = pd.merge(vendas_por_estado, df_metas, on='estado_cliente', how='inner')
df_merge

### Exercício 9 (Operações Vetoriais / Broadcasting)
1. Diferença entre faturamento real e meta (`faturamento_real - meta_faturamento`)
2. Porcentagem de atingimento (`(faturamento_real / meta_faturamento) * 100`)

In [ ]:
# 1. Diferença absoluta (Broadcasting)
df_merge['diferenca_meta'] = df_merge['faturamento_real'] - df_merge['meta_faturamento']

# 2. Porcentagem de atingimento (Broadcasting)
df_merge['pct_atingimento'] = (df_merge['faturamento_real'] / df_merge['meta_faturamento']) * 100

# Formatando a visualização final
df_merge_final = df_merge.sort_values(by='pct_atingimento', ascending=False)
df_merge_final